# API & Demo Walkthrough

**Objective:** Document the FastAPI inference service and Streamlit demo built on top of the NB07 ensemble — what each component does, why it was built this way, and how to run it locally.

**Author:** Ayush Singh

## 1. Why a Serving Layer

NB08 showed that the inference pipeline works end-to-end inside a notebook — a single record can be scored, sensitivity sweeps can be run, and batch predictions can be saved to CSV. But a notebook is not how you deliver predictions outside the data science environment. To turn the NB07 ensemble into something usable, two components were added.

**FastAPI service (`src/api.py`)** exposes the model as an HTTP endpoint. Any application — a web frontend, a backend service, a data pipeline, a curl command — can send a JSON payload and receive a calibrated click probability without knowing anything about the model internals. Pydantic validates every field on the way in, so the model never receives malformed input.

**Streamlit app (`src/app.py`)** is an interactive demo. It reproduces the NB08 sensitivity analyses — position sweep and HistCTR sweep — as live charts that update as you move the sliders. This makes the model's behaviour tangible without needing to read notebook output.

**Shared module (`src/feature_engineering.py`)** is the architectural decision that makes both of the above safe. Both the API and the app import `engineer_features()` from this single file. The feature logic is defined exactly once. This avoids the classic training-serving skew problem: model trained on features computed one way, predictions served on features computed slightly differently. If the feature logic ever changes, there is one place to update it.

## 2. Repository Structure — `src/`

```
src/
├── __init__.py
├── feature_engineering.py   # shared feature logic
├── api.py                   # FastAPI service
└── app.py                   # Streamlit demo
```

**`__init__.py`** — empty file that makes `src/` a Python package. Required so that `from src.feature_engineering import ...` resolves correctly when the API or app is launched from the repo root.

**`feature_engineering.py`** — defines `FEATURE_COLS`, the three smoothing constants, and `engineer_features(record)`. This is the single source of truth for how raw impression fields are converted into model inputs; neither the API nor the app contains any feature logic of their own.

**`api.py`** — FastAPI application with three endpoints: `/health`, `/predict`, and `/batch_predict`. It loads the four saved models once at startup using `joblib`, defines the Pydantic input and output schemas, and delegates all feature transformation to `engineer_features()`.

**`app.py`** — Streamlit single-page application with a two-column layout: input widgets on the left, live prediction metric and two sensitivity charts on the right. It loads models once via `@st.cache_resource` and calls `engineer_features()` directly — no dependency on the API being running.

## 3. Feature Engineering Module

`src/feature_engineering.py` contains everything needed to convert a raw impression dictionary into a model-ready feature row.

**Constants:**
- `FEATURE_COLS` — the 21-element list of feature names in the exact column order NB05 produced and NB06–08 trained on. Column order matters because the models were saved with named features; passing a DataFrame with columns in the wrong order would silently produce wrong predictions.
- `GLOBAL_CTR = 0.006142` — the contextual click rate from NB04, used as the fallback when a user has no prior impression history.
- `ALPHA = 0.05`, `BETA = 75` — the Laplace smoothing parameters transferred from KDD Cup NB02 and used throughout NB05.

**`engineer_features(record: dict) -> pd.DataFrame`** takes any dictionary of raw fields and returns a `(1, 21)` DataFrame. It handles three fallback cases:
- **User history absent or zero** → `user_historical_ctr` falls back to `GLOBAL_CTR` (0.0061)
- **Entity rate encoding absent or `None`** → falls back to `smoothed_ctr(0, 0)` (the Laplace prior for an unseen entity)
- **`SearchDate` absent or `None`** → falls back to `pd.Timestamp.now()`

The `None`-check fallback matters in practice: when the FastAPI endpoint calls `request.dict()`, Pydantic includes every optional field explicitly as `None`. A plain `dict.get(key, default)` finds the key and returns `None` rather than triggering the default, so each fallback must be an explicit `is not None` check rather than a `.get()` default.

In [ ]:
# Illustrative — not executed in this notebook
from src.feature_engineering import engineer_features

record = {
    "HistCTR": 0.008,
    "Position": 1,
    "Price": 4500,
    "Title": "Продам ноутбук Lenovo ThinkPad",
    "IsUserLoggedOn": 1,
    "SearchDate": "2015-06-01 14:00:00",
}
X = engineer_features(record)
# Returns a 1×21 DataFrame ready for model.predict_proba()
print(X.shape)          # (1, 21)
print(X.columns.tolist())

## 4. FastAPI Service

### Why FastAPI

FastAPI generates OpenAPI documentation automatically at `/docs` — every field, its type, its constraints, and the example payload are all visible and testable in a browser without writing a single line of documentation by hand. Pydantic handles input validation before the request touches any model code: a payload with `Position=99` or `HistCTR="not a number"` returns a structured 422 error with the exact field and reason, rather than a `TypeError` buried inside `predict_proba`. The framework is also async-ready for production scaling without a rewrite.

### Endpoints

| Endpoint | Method | Purpose |
|---|---|---|
| `/health` | GET | Liveness check — confirms models are loaded, returns ensemble AUC and log-loss |
| `/predict` | POST | Score a single `AdRequest` → `PredictionResponse` |
| `/batch_predict` | POST | Score up to 10,000 records in one call → `BatchResponse` |

### Running the API

```bash
# From repo root with venv active
uvicorn src.api:app --reload --host 0.0.0.0 --port 8000
```

The `--reload` flag restarts the server on file changes, useful during development. Drop it for a stable run.

### Health check

```bash
curl http://localhost:8000/health
# {"status":"ok","models_loaded":["xgb_avito","platt_xgb","lgb_avito","platt_lgb"],"ensemble_auc":0.7613,"ensemble_log_loss":0.03404}
```

### Single-record prediction

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "HistCTR": 0.008,
    "Position": 1,
    "IsUserLoggedOn": 1,
    "Price": 4500,
    "Title": "Продам ноутбук Lenovo ThinkPad",
    "category_level": 2,
    "category_match": 1,
    "session_size": 2,
    "user_impression_count": 12,
    "user_click_count": 2,
    "uid_category_count": 3,
    "ad_ctr": 0.009,
    "category_ctr": 0.007,
    "location_ctr": 0.006,
    "position_ctr": 0.006,
    "device_ctr": 0.006,
    "SearchDate": "2015-06-01 14:00:00"
  }'
```

Expected response shape:

```json
{
  "p_click": 0.007565,
  "p_click_pct": 0.7565,
  "xgb_calibrated": 0.006775,
  "lgb_calibrated": 0.008355,
  "baseline_ctr": 0.006142,
  "above_baseline": true
}
```

### Interactive docs

Visiting `http://localhost:8000/docs` opens the auto-generated Swagger UI. Every endpoint is listed with its full input schema, field constraints, and the example payload pre-filled. The "Try it out" button sends a live request and displays the response — no curl needed for manual testing.

## 5. Streamlit Demo

`src/app.py` is a single-page Streamlit application that makes the model interactive.

**Why Streamlit:** the entire UI is Python — no HTML, no JavaScript, no frontend framework. `@st.cache_resource` loads the four model files once the first time the app starts and holds them in memory across all subsequent reruns. Every widget interaction triggers a full Python rerun, so the prediction, the position sweep, and the HistCTR sweep all update in real time without explicit callback wiring.

**Layout:** two-column design. The left column contains all input controls — HistCTR slider, position selector, session size, login status, category match, price, title, category level, user history counts, and the three rate feature sliders. The right column displays the live prediction metric, above/below baseline status, the model breakdown table, and the two sensitivity charts.

**Sensitivity charts:** the position sweep (positions 1–7) and the HistCTR sweep (five values from 0.001 to 0.020) reproduce the NB08 sensitivity analyses interactively. The position gradient inversion — where the model predicts Pos7 > Pos1 due to the session_size confound — and the non-monotone HistCTR response are visible in real time as you adjust the input sliders. The charts update with every widget change, making both anomalies easy to demonstrate.

**No API dependency:** the Streamlit app calls `engineer_features()` and `model.predict_proba()` directly. It does not make HTTP requests to the FastAPI service. Both components are independent — you can run the app without the API running, or run the API without the app.

### Running the demo

```bash
# From repo root with venv active
streamlit run src/app.py --server.port 8003
# Then open http://localhost:8003
```

## 6. Design Decisions

**1. Why not Docker?**

This is a portfolio demo running on a shared GPU server where I do not have root access. Docker adds real operational complexity — image builds, volume mounts, port mapping, container lifecycle — with no benefit for a local demonstration. The dependencies are fully specified in `requirements_api.txt`; anyone cloning the repo can create a venv and install them in under a minute. Containerisation would be the right next step before deploying to a shared or cloud environment.

**2. Why no authentication on the API?**

The `/predict` endpoint accepts requests from any caller with no credentials. That is appropriate for a portfolio demo but would be wrong in production. A real deployment would add an API key in an `Authorization` header, validate it with FastAPI's dependency injection system, and add rate limiting per key. The Pydantic schema and endpoint structure are compatible with adding this without a rewrite.

**3. Why Streamlit over Gradio?**

Both libraries produce browser-based ML demos in pure Python. Gradio is better suited to single-function interfaces: one set of inputs, one output, done. Streamlit handles multi-panel layouts and arbitrary chart types more naturally. This app has three output sections (metric, position sweep chart, HistCTR sweep chart) that need to update together and be laid out side-by-side — Streamlit's column and `st.line_chart` primitives handle that more cleanly than Gradio's block API.

**4. Why a shared `feature_engineering.py`?**

Training-serving skew is the single biggest operational risk in a deployed ML system. The model was trained on features computed a specific way — Laplace smoothing with α=0.05, β=75; `shift(1)` for cumulative user statistics; `np.log1p` for price. If the serving layer recomputes those features with any difference — wrong smoothing constant, wrong fill value, wrong column order — the model receives inputs it was never trained on and its calibration breaks silently. A single importable module with one implementation eliminates this class of error entirely.

## 7. Limitations and What I Would Do Next

**1. Out-of-distribution inputs are accepted silently.** The API rejects structurally invalid inputs (wrong type, out of range) via Pydantic. But it accepts semantically implausible inputs without warning — `HistCTR=0.999` passes the `ge=0, le=1` constraint but is wildly outside the training distribution. A statistical OOD check (e.g., flagging inputs more than 3 standard deviations from training feature means) would catch these and return a warning alongside the prediction.

**2. XGBoost Platt calibration is impure.** The Platt scaler for XGBoost was fit on the 60–80% chronological slice of the data, which was inside NB06's 80% training window. The logistic regression therefore fitted on data the base model had already seen, which overstates calibration quality. The fix is to retrain XGBoost on 60% only (matching the LightGBM setup in NB07) and refit the Platt scaler on a genuinely held-out 20% fold. LightGBM's calibration is fully clean and is the more trustworthy baseline.

**3. No model versioning.** The `joblib` files are loaded by hardcoded filename. A retrained model silently overwrites the previous one with no audit trail, no rollback, and no A/B testing capability. A production system would load models by version tag from a model registry (MLflow, BentoML, or similar) and support rollback to any previous version without code changes.

**4. Hardcoded `SearchDate` in the Streamlit app.** The app passes `SearchDate = '2015-06-01 14:00:00'` to match the training distribution. The models were trained on 2015 Avito data, so feeding a 2026 timestamp produces hour-of-day and day-of-week features that are structurally valid but temporally shifted from the training set. In production the actual request timestamp would be used, but temporal distribution shift in the `hour_of_day` and `day_of_week` features would need to be monitored.

**5. No explanation endpoint.** The API returns a calibrated click probability but gives no indication of which features drove it. Adding a `/explain` endpoint using SHAP values would return the top 3 features contributing to the prediction for each record — making the model's reasoning visible to downstream consumers and simplifying debugging when predictions look surprising.